# Banking feedback: baseline preparation

Prepare conventional classifiers on existing source ratings while full-corpus labels are processed.


## External inputs and source


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path
started=time.monotonic()
notebook=next(p for p in [Path('__notebook__.ipynb'),Path('/kaggle/working/__notebook__.ipynb')] if p.exists())
submitted=json.loads(notebook.read_text())
payload=next(c['attachments']['evaluation_assets.zip']['application/zip'] for c in submitted['cells'] if 'evaluation_assets.zip' in c.get('attachments',{}))
root=Path('/kaggle/working/evaluation_assets')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist()), 'UNSAFE_ASSET_PATH'
    manifest=json.loads(z.read('checksums.json'))
    assert set(z.namelist())=={v['path'] for v in manifest}|{'checksums.json'}, 'ASSET_MEMBER_MISMATCH'
    for item in manifest:
        raw=z.read(item['path'])
        assert len(raw)==item['bytes'] and hashlib.sha256(raw).hexdigest()==item['sha256'], 'ASSET_HASH_MISMATCH'
    z.extractall(root)
os.environ['EVALUATION_ASSET_ROOT']=str(root)
sys.path.insert(0,str(root))
print('RESOLVED_ASSETS',str(root),len(manifest))


## Executable source checks


In [ ]:
from source_audit import source_audit
audits=[source_audit(p) for p in root.glob('*.py')]
directory=Path('/kaggle/working/browser_source_checks')
directory.mkdir(exist_ok=True)
cell_audits=[]
for i,cell in enumerate(submitted['cells']):
    if cell['cell_type']=='code':
        p=directory/f'cell_{i:03d}.py'
        p.write_text(''.join(cell['source']))
        cell_audits.append(source_audit(p))
(directory/'source_audits.json').write_text(json.dumps(audits,indent=2))
(directory/'notebook_code_audit.json').write_text(json.dumps(cell_audits,indent=2))
print('SOURCE_AUDITS',json.dumps(audits))
print('NOTEBOOK_SOURCE_AUDIT',json.dumps(cell_audits))


## Fixed split and classifier preparation


In [ ]:
try:
    subprocess.run([sys.executable,str(root/'pretrain_baselines.py')],check=True)
finally:
    print('TOTAL_NOTEBOOK_SECONDS',round(time.monotonic()-started,3))


## Retained preparation record


In [ ]:
summary=json.loads(Path('/kaggle/working/baseline_pretraining/pretraining.json').read_text())
print(json.dumps(summary,indent=2))
